In [3]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
DEFAULT_MODEL = "claude-haiku-4-5-20251001"


def add_user_message(messages: list[dict], text: str) -> None:
    """Appends a user message to the conversation history."""
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages: list[dict], text: str) -> None:
    """Appends an assistant message to the conversation history."""
    messages.append({"role": "assistant", "content": text})


def chat(
    messages: list[dict],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
    model: str = DEFAULT_MODEL,
) -> str:
    """Sends the message history to Claude and returns the response text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()
client = Anthropic()

DEFAULT_MODEL = "claude-haiku-4-5-20251001"


class TaskItem(BaseModel):
    task: str = Field(
        description="Description of an AWS task solvable by Python, JSON, or Regex"
    )


class TaskDataset(BaseModel):
    tasks: list[TaskItem]


def generate_dataset(model: str = DEFAULT_MODEL) -> TaskDataset:
    prompt = """
    Generate an evaluation dataset for prompts that produce Python, JSON, or Regex for AWS tasks.
    
    Requirements:
    - Tasks must be solvable by a single Python function, JSON object, or Regex.
    - Tasks should require minimal code.
    - Generate 3 distinct objects.
    """

    response = client.messages.parse(
        model=model,
        max_tokens=1024,
        output_format=TaskDataset,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.parsed_output


dataset = generate_dataset()
print(dataset.model_dump_json(indent=2))

{
  "tasks": [
    {
      "task": "Write a Python function that parses an AWS S3 bucket name from a full S3 ARN string like 'arn:aws:s3:::my-bucket/path/to/object' and returns just the bucket name."
    },
    {
      "task": "Create a JSON object representing an AWS IAM policy that grants read-only access to all objects in an S3 bucket named 'data-lake'."
    },
    {
      "task": "Write a Regex pattern that validates AWS EC2 instance IDs, which follow the format 'i-' followed by 8 or 17 hexadecimal characters (e.g., 'i-0abcd1234efgh5678')."
    }
  ]
}


In [6]:
import json

with open("dataset.json", "w") as f:
    json.dump(dataset.model_dump(), f, indent=2)

In [18]:
def run_prompt(test_case: dict, model: str = DEFAULT_MODEL) -> str:
    """Merges the prompt and test case input, then executes the chat call."""
    task_description = test_case.get("task", "")

    prompt = f"""Please solve the following task:

{task_description}"""

    messages = []
    add_user_message(messages, prompt)

    return chat(messages, model=model)

In [ ]:
def run_test_case(test_case: dict, model: str = DEFAULT_MODEL) -> dict:
    """Calls run_prompt, grades the resulting output, and returns a summary dict."""
    output = run_prompt(test_case, model=model)

    # TODO: Implement your custom grading logic here
    # (e.g., executing the Python code in a safe sandbox, validating JSON, or testing regex)
    score = 10

    return {"output": output, "test_case": test_case, "score": score}

In [16]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [17]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(results)

TypeError: string indices must be integers, not 'str'